# Amazon ML Challenge 2026 — Business Entity Resolution

## Setup (do ONCE before running)
1. **Add Data** (right panel) → Your dataset: `rohithpanchamukhi/amazon-ml-2026-dataset`
2. **Accelerator** → GPU T4 x2
3. **Internet** → ON (needed for git clone)
4. Click **Run All**

In [ ]:
# Cell 1: Environment setup
import os, sys, glob, shutil, subprocess, zipfile

KAGGLE = os.path.exists('/kaggle/working')
print('Running on Kaggle:', KAGGLE)

WORK_DIR   = '/kaggle/working' if KAGGLE else os.getcwd()
DATA_DIR   = os.path.join(WORK_DIR, 'dataset')
OUTPUT_DIR = os.path.join(WORK_DIR, 'output')

# Kaggle mounts datasets at /kaggle/input/<dataset-slug>/
# The slug is the last part of the dataset URL
DATASET_SLUG = 'amazon-ml-2026-dataset'   # <-- your dataset slug
KAGGLE_INPUT = f'/kaggle/input/{DATASET_SLUG}'

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'WORK_DIR   : {WORK_DIR}')
print(f'DATA_DIR   : {DATA_DIR}')
print(f'OUTPUT_DIR : {OUTPUT_DIR}')
print(f'Input path : {KAGGLE_INPUT}')

if KAGGLE:
    print('\nFiles visible in /kaggle/input:')
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            fp = os.path.join(root, f)
            print(f'  {fp}  ({os.path.getsize(fp)/1024**2:.1f} MB)')

In [ ]:
# Cell 2: Extract dataset
# On Kaggle, zip files in datasets are NOT auto-extracted — we do it here.

# Find the zip inside the mounted dataset
zip_candidates = glob.glob(f'{KAGGLE_INPUT}/**/*.zip', recursive=True) \
               + glob.glob(f'{KAGGLE_INPUT}/*.zip')

print(f'Searching for zip in: {KAGGLE_INPUT}')
print(f'Found: {zip_candidates}')

if not zip_candidates:
    raise FileNotFoundError(
        f'No zip found under {KAGGLE_INPUT}.\n'
        f'Make sure you added the dataset "{DATASET_SLUG}" via Add Data.'
    )

zip_path = zip_candidates[0]
print(f'\nExtracting: {zip_path}')

# Check if already extracted (re-run safety)
already_done = os.path.exists(os.path.join(DATA_DIR, 'train', 'train_source1.tsv'))
if already_done:
    print('Dataset already extracted — skipping.')
else:
    with zipfile.ZipFile(zip_path, 'r') as z:
        # Only extract dataset files, skip __MACOSX and .DS_Store
        entries = [
            e for e in z.namelist()
            if 'student_resource/dataset/' in e
            and '__MACOSX' not in e
            and '.DS_Store' not in e
        ]
        print(f'Entries to extract: {len(entries)}')

        for entry in entries:
            # Strip leading 'student_resource/dataset/' prefix
            rel = entry.replace('student_resource/dataset/', '', 1)
            if not rel:
                continue
            dest = os.path.join(DATA_DIR, rel)
            if entry.endswith('/'):
                os.makedirs(dest, exist_ok=True)
            else:
                os.makedirs(os.path.dirname(dest), exist_ok=True)
                with z.open(entry) as src, open(dest, 'wb') as dst:
                    shutil.copyfileobj(src, dst)
                print(f'  {rel}  ({os.path.getsize(dest)/1024**2:.0f} MB)')

# Confirm all 7 files present
print('\nExtracted dataset files:')
for root, dirs, files in os.walk(DATA_DIR):
    for f in sorted(files):
        fp = os.path.join(root, f)
        print(f'  {fp}  ({os.path.getsize(fp)/1024**2:.1f} MB)')

In [ ]:
# Cell 3: Clone the code repo (enhanced-pipeline branch)
REPO_URL = 'https://github.com/ROHITH-KUMAR-L/amazon-ml.git'
BRANCH   = 'improvements/enhanced-pipeline'
REPO_DIR = os.path.join(WORK_DIR, 'amazon-ml')

if not os.path.exists(REPO_DIR):
    print(f'Cloning {REPO_URL} @ {BRANCH} ...')
    result = subprocess.run(
        ['git', 'clone', '--depth=1', '--branch', BRANCH, REPO_URL, REPO_DIR],
        capture_output=True, text=True
    )
    print(result.stdout or 'Clone complete.')
    if result.returncode != 0:
        print('STDERR:', result.stderr)
        raise RuntimeError('Git clone failed — make sure Internet is ON in Kaggle settings.')
else:
    print('Repo already cloned, pulling latest...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], capture_output=True)

# Add to Python path
SRC_ROOT = os.path.join(REPO_DIR, 'student_resource', 'code', 'business_entity_resolution')
if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

print(f'\nSource root: {SRC_ROOT}')
print('src/ contents:', os.listdir(os.path.join(SRC_ROOT, 'src')))

In [ ]:
# Cell 4: Install / upgrade dependencies
# Kaggle already ships: pandas, numpy, sklearn, scipy, lightgbm, xgboost, torch (CUDA)
# We pin versions to match requirements.txt
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'lightgbm>=4.0.0', 'catboost>=1.2.0', 'xgboost>=2.0.0', 'tqdm>=4.65.0'],
    check=True
)

import lightgbm, catboost, xgboost, sklearn, scipy, torch
print('Versions:')
print(f'  LightGBM : {lightgbm.__version__}')
print(f'  CatBoost : {catboost.__version__}')
print(f'  XGBoost  : {xgboost.__version__}')
print(f'  Sklearn  : {sklearn.__version__}')

print(f'\nCUDA available: {torch.cuda.is_available()}')
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {props.name}  {props.total_memory/1024**3:.1f} GB VRAM')

In [ ]:
# Cell 5: Run the full pipeline
import time
t_start = time.time()

from src.pipeline import run_pipeline

run_pipeline(
    data_dir          = DATA_DIR,
    output_dir        = OUTPUT_DIR,
    sample_train_size = 0,       # 0 = ALL training data (Kaggle has 30 GB RAM)
    use_gpu           = True,
    neg_sample_ratio  = 10.0,
)

elapsed = time.time() - t_start
print(f'Total time: {elapsed/60:.1f} minutes')

In [ ]:
# Cell 6: Validate submission
VALIDATOR = os.path.join(REPO_DIR, 'student_resource', 'utils', 'validate_submission.py')
TEST_DIR  = os.path.join(DATA_DIR, 'test')

result = subprocess.run(
    [
        sys.executable, VALIDATOR,
        '--matching',  os.path.join(OUTPUT_DIR, 'matching_results.tsv'),
        '--candidate', os.path.join(OUTPUT_DIR, 'candidate_pairs.tsv'),
        '--test-dir',  TEST_DIR,
    ],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('ERRORS:', result.stderr)
else:
    print('VALIDATION PASSED -- Safe to submit!')

In [ ]:
# Cell 7: Preview output stats
import pandas as pd

matching   = pd.read_csv(os.path.join(OUTPUT_DIR, 'matching_results.tsv'),
                         sep='\t', keep_default_na=False)
candidates = pd.read_csv(os.path.join(OUTPUT_DIR, 'candidate_pairs.tsv'),
                         sep='\t', keep_default_na=False)

has_match = matching['matched_entity_ids'].str.len() > 0
print('=== matching_results.tsv ===')
print(f'  Total S1 rows      : {len(matching):,}')
print(f'  Matched entities   : {has_match.sum():,}')
print(f'  Singletons (empty) : {(~has_match).sum():,}')
print(matching.head(10).to_string(index=False))

print('\n=== candidate_pairs.tsv ===')
print(f'  Total rows: {len(candidates):,}')
print(candidates.head(5).to_string(index=False))

print('\nOutput files:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    fp = os.path.join(OUTPUT_DIR, f)
    print(f'  {f}  ({os.path.getsize(fp)/1024**2:.1f} MB)')

## Download & Submit
1. **Output tab** (right panel) → `output/matching_results.tsv` → Download
2. Upload `matching_results.tsv` to the Amazon ML Challenge portal
3. For final zip submission, also include `candidate_pairs.tsv`